A sematic, structure-aware Markdown chunker pipeline. It converts a markdown into a list of enriched `Chunk` object ready to be embedded and stored in a vector database.

In [19]:
import hashlib
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional
import tiktoken
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt', quiet=True)

True

In [20]:
_TOKENIZER = tiktoken.get_encoding("o200k_base") # tokenizer for openai models
def count_tokens(text: str) -> int:
    """Return the exact token count for *text* using tiktoken."""
    return len(_TOKENIZER.encode(text))

In [21]:
count_tokens("Hi my name is Long. I am from Hanoi Vietnam")

11

In [22]:
# Each chunk will be an object of Chunk class
@dataclass
class Chunk:
    """
    chunk_id: Stable SHA-256 fingerprint. Used for primary key in a vector store.
            Same id will be use for identical text, so re-indexing a document is idempotent (unchanged)
    
    doc_title: Document title derive from document filename
    
    source_file: Absolute or relative path
    
    section_path: Breadcrumb of heading. Empty string if no heading.
                    E.g.
                    ```
                    # Chapter 3
                    ## Retrieval
                    ### Dense Retrieval
                    ```
                    It will be "Chapter 3 > Retrieval > Dense Retrieval"
                    
    prev_heading: The parent heading one level up from the current section.
                    Useful for adding context.
                
    chunk_type: One of: "paragraph", "bullet_list", "table", "code_block",
                "mixed", "empty".
    
    chunk_text: Raw chunk text
    
    embedding_text: chunk_text after processed with metadata. Feed this to
                    embedding model, not chunk_text.
    
    token_count: Cached token count of embedding_text, so callers don't need to
                re-tokenize when building batches. 
    
    """
    chunk_id: str
    doc_title: str
    source_file: str
    section_path: str
    prev_heading: Optional[str]
    chunk_type: str
    chunk_text: str
    embedding_text: str
    token_count: int = field(default=0)

In [23]:
# Matches any markdown heading: # … through ###### …
HEADING_RE = re.compile(r"^(#{1,6})\s+(.+)$")

# Matches a line that starts a bullet or numbered list item
LIST_ITEM_RE = re.compile(r"^\s*([-*+]|\d+\.)\s+")

# Matches a markdown table row (contains at least one pipe character)
TABLE_ROW_RE = re.compile(r"^\|.+\|")

In [ ]:
def parse_markdown_sections(markdown_text: str) -> List[dict]:
    """
    Split a markdown document into a list of sections
    
    Each section is the content the falls under a particular heading. 
    The function tracks:
        - section_path: breadcrumb heading path
        - prev_heading: Immediate closing parent heading
    
    Returns a list of dicts each with keys: section_path, prev_heading, content
    """
    
    lines = markdown_text.splitlines()
    sections: List[dict] = []
    
    # Track hierarchy of current heading. 
    # e.g., [(1, "Chapter 1"), (2, "Section A")] = "Chapter 1 > Section A"
    heading_stack: List[tuple] = []

    # Accumulate lines until we hit the next heading
    current_lines: List[str] = []
    
    def flush_section() -> None:
        """Push the accumulated lines as a completed section."""
        nonlocal current_lines
        
        # Skip empty section
        content = "\n".join(current_lines).strip()
        if not content:
            current_lines = []
            return

        # Build the breadcrumb path from the heading stack
        section_path = " > ".join(h[1] for h in heading_stack)
        
        # Extract parent heading (one level up in hierarchy)
        # Useful when current heading is vague like "Overview" or "Introduction"
        
        prev_heading = heading_stack[-2][1] if len(heading_stack) >= 2 else None

        sections.append(
            {
                "section_path": section_path,
                "prev_heading": prev_heading,
                "content": content,
            }
        )
        
        current_lines = []
        
    # Process each line
    for line in lines:
        
        # Check for markdown headings like # or ## or ###
        heading_match = HEADING_RE.match(line)
        if heading_match:
            # New heading, finalize the previous section first.
            flush_section()

            # Extract the heading level and text
            level = len(heading_match.group(1))  
            heading_text = heading_match.group(2).strip()
            
            # Update the hierarchy: pop headings at same/deeper level
            # This ensures heading_stack is always a valid ancestor chain
            while heading_stack and heading_stack[-1][0] >= level:
                heading_stack.pop()

            heading_stack.append((level, heading_text))
        else:
            current_lines.append(line)
    
    flush_section()
    
    return sections

In [25]:
file_path = r"D:\Deakin-Data-Science\T2Y3\SIT378 - Team project B\Project\Project 6 - Bridge and Road Crack Detection System\RAG pipeline\processed_documents\Austroads Guide to Bridge Technology Part 7.md"
with open(file_path, "r", encoding="utf-8") as f:
    markdown_text = f.read()

sections = parse_markdown_sections(markdown_text)
print(f"Total sections found: {len(sections)}")

Total sections found: 486


In [26]:
sections[:3]

[{'section_path': '1. Introduction > 1.1 Scope',
  'prev_heading': '1. Introduction',
  'content': "Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on the maintenance and management of existing bridges and addresses topics such as:\n\n- bridge inspection\n- load rating and monitoring\n- maintenance, repair and strengthening\n- bridge preservation.\n\nPart 7 provides documentation of the technical issues and practices relevant to the above topics and is intended to present good practice in the technical aspects of bridge management consistent with Integrated Asset Management Guidelines for Road Networks (Austroads 2002). Part 7 can be used to complement Guide to Asset Management Technical Information Part 13: Structures Asset Management (Austroads 2018), which provides guidance on the establishment and maintenance of structure asset inventories and the monitoring of asset performance.\n\nThe focus of Part 7 is on the practical and technical procedures and str

In [27]:
for i, section in enumerate(sections[:5], 1):  # Show first 5
    print(f"--- Section {i} ---")
    print(f"Path: {section['section_path']}")
    print(f"Parent: {section['prev_heading']}")
    print(f"Content preview: {section['content'][:150]}...\n")

--- Section 1 ---
Path: 1. Introduction > 1.1 Scope
Parent: 1. Introduction
Content preview: Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on the maintenance and management of existing bridges and addresses topics...

--- Section 2 ---
Path: 1. Introduction > 1.2 Guide Structure
Parent: 1. Introduction
Content preview: The AGBT is published in eight parts and addresses a range of bridge technology issues, each of which is summarised in Table 1.1.

Table 1.1: Parts of...

--- Section 3 ---
Path: 2. Maintenance and Management of Existing Bridges > 2.1 Introduction
Parent: 2. Maintenance and Management of Existing Bridges
Content preview: Bridges are key elements in any road network and represent a major investment of community resources which has been built up progressively over time. ...

--- Section 4 ---
Path: 2. Maintenance and Management of Existing Bridges > 2.2 General Administration and Management Systems
Parent: 2. Maintenance and Management of Exis

In [ ]:
def split_into_blocks(section_text: str) -> List[str]:
    """
    Each section is split into sematic blocks. 
    A block is group of lines separated from its neighbor by one or more blank lines.
    Except fenced code blocks, we use ``` ... ```.
    
    Returns a list of non-empty block strings
    """
    lines = section_text.splitlines()
    blocks: List[str] = []
    current_block: List[str] = []
    in_code_fence = False  # True while we're inside a ``` … ``` pair
    
    for line in lines:
        # Code-fence flag on for code block
        if line.strip().startswith("```"):
            in_code_fence = not in_code_fence
            current_block.append(line)
            continue
        
        if not in_code_fence and not line.strip():
            # Blank line and not code block -> end of current block
            if current_block:
                blocks.append("\n".join(current_block).strip())
                current_block = []
        else:
            current_block.append(line)
        
    # Flush the last block
    if current_block:
        blocks.append("\n".join(current_block).strip())

    non_empty = [b for b in blocks if b.strip()]
    return non_empty

In [29]:
# Get the first section of markdown_text
section_text = markdown_text[markdown_text.find("## 1.1 Scope"):markdown_text.find("## 1.2 Guide Structure")]

# Split into blocks
blocks = split_into_blocks(section_text)

print(f"Total blocks: {len(blocks)}\n")

Total blocks: 21



In [30]:
print(section_text[:100])

## 1.1 Scope

Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on the mai


In [31]:
# Display first 5 blocks
for i, block in enumerate(blocks[:5], 1):
    print(f"--- Block {i} ---")
    preview = block[:500].replace('\n', ' ')
    print(f"Preview: {preview}...")
    print(f"Length: {len(block)} chars\n")

--- Block 1 ---
Preview: ## 1.1 Scope...
Length: 12 chars

--- Block 2 ---
Preview: Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on the maintenance and management of existing bridges and addresses topics such as:...
Length: 159 chars

--- Block 3 ---
Preview: - bridge inspection - load rating and monitoring - maintenance, repair and strengthening - bridge preservation....
Length: 111 chars

--- Block 4 ---
Preview: Part 7 provides documentation of the technical issues and practices relevant to the above topics and is intended to present good practice in the technical aspects of bridge management consistent with Integrated Asset Management Guidelines for Road Networks (Austroads 2002). Part 7 can be used to complement Guide to Asset Management Technical Information Part 13: Structures Asset Management (Austroads 2018), which provides guidance on the establishment and maintenance of structure asset inventori...
Length: 543 chars

--- Block 5 ---
Preview: Th

In [32]:
def normalize_whitespace(text: str) -> str:
    """
    This function cleans blank lines in blocks by
    collapsing runs of 3+ consecutive blank lines down to exactly two.

    This keeps intentional paragraph breaks without letting runaway blank
    lines bloat token counts.
    """
    return re.sub(r"\n{3,}", "\n\n", text).strip()

In [33]:
sample = """
Hi my name is Long Tran.




My favorite sport is Football.
"""

normalized = normalize_whitespace(sample)
print("=== BEFORE ===")
print(sample)
print("\n=== AFTER ===")
print(normalized)

=== BEFORE ===

Hi my name is Long Tran.




My favorite sport is Football.


=== AFTER ===
Hi my name is Long Tran.

My favorite sport is Football.


In [34]:
def detect_chunk_type(block: str) -> str:
    """
    Classify a block into one of five sematic types.
    
    "code_block"  — fenced code (``` … ```)
    "table"       — markdown table rows
    "bullet_list" — bullet or numbered list items
    "paragraph"   — continuous prose
    "mixed"       — block contains more than one of the above
    "empty"       — nothing here
    """
    stripped = block.strip()

    if not stripped:
        return "empty"
    
    # Code block
    lines = stripped.splitlines()
    if lines[0].strip().startswith("```") and lines[-1].strip().startswith("```"):
        return "code_block"
    
    # Dictionary counting types
    type_counts: dict[str, int] = {
        "table": 0,
        "list": 0,
        "prose": 0,
    }
    
    for line in lines:
        if not line.strip():
            continue  # blank separators don't count
        if TABLE_ROW_RE.match(line):
            type_counts["table"] += 1
        elif LIST_ITEM_RE.match(line):
            type_counts["list"] += 1
        else:
            type_counts["prose"] += 1
            
    # Determine how many distinct types appear
    types_count = [t for t, n in type_counts.items() if n > 0]
    
    if len(types_count) > 1:
        return "mixed"
    
    if type_counts["table"] > 0:
        return "table"
    if type_counts["list"] > 0:
        return "bullet_list"

    return "paragraph"

In [35]:
for i, block in enumerate(blocks[:10], 1):
    chunk_type = detect_chunk_type(block)
    preview = block[:60].replace("\n", " ")
    print(f"Block {i}: {chunk_type}")
    print(f"  Preview: {preview}...\n")

Block 1: paragraph
  Preview: ## 1.1 Scope...

Block 2: paragraph
  Preview: Part 7 of the Austroads Guide to Bridge Technology (AGBT) pr...

Block 3: bullet_list
  Preview: - bridge inspection - load rating and monitoring - maintenan...

Block 4: paragraph
  Preview: Part 7 provides documentation of the technical issues and pr...

Block 5: paragraph
  Preview: The focus of Part 7 is on the practical and technical proced...

Block 6: paragraph
  Preview: The target audience includes all those involved with the man...

Block 7: paragraph
  Preview: The bridge manager is faced with many choices in the allocat...

Block 8: paragraph
  Preview: It is accepted that bridges represent a significant but dist...

Block 9: paragraph
  Preview: The management of existing bridges involves dealing with the...

Block 10: paragraph
  Preview: The Australian experience is of a diverse history and practi...



In [ ]:
def split_block_by_tokens(
    block: str,
    max_tokens: int = 4000,
    overlap_tokens: int = 40,
) -> List[str]:
    """
    Further split a block if it exceeds max_token into smaller chunks
    using a sentence-level sliding window. I am using gpt5 nano model with
    400,000 token context window, so I decide to set the max_token to high.
    The focus is not constrain but retrieval quality.
    
    Args:
        block         : The text to split.
        max_tokens    : Hard upper bound (in tokens) for each output chunk.
        overlap_tokens: How many tokens of the *previous* chunk to prepend to the
                        next one. 
    """
    if count_tokens(block) <= max_tokens:
        return [block]
    
    # Split on sentence boundaries
    sentences = sent_tokenize(block.strip())
    
    chunks: List[str] = []
    current_sentences: List[str] = []
    current_token_count = 0
    
    for sentence in sentences:
        sentence_tokens = count_tokens(sentence)

        if current_token_count + sentence_tokens <= max_tokens:
            # Sentence fits — add it to the running chunk.
            current_sentences.append(sentence)
            current_token_count += sentence_tokens
        else:
            # Sentence would overflow — seal the current chunk first.
            if current_sentences:
                chunks.append(" ".join(current_sentences).strip())

            # Walk backwards through the sentences of the just-sealed chunk
            # and collect as many as fit within overlap_tokens.
            overlap_sentences: List[str] = []
            overlap_used = 0

            for prev_sentence in reversed(current_sentences):
                prev_tokens = count_tokens(prev_sentence)
                if overlap_used + prev_tokens <= overlap_tokens:
                    overlap_sentences.insert(0, prev_sentence)
                    overlap_used += prev_tokens
                else:
                    break

            # Start the new chunk with the overlap tail + the current sentence.
            current_sentences = overlap_sentences + [sentence]
            current_token_count = sum(count_tokens(s) for s in current_sentences)

    # Seal any remaining sentences as the final chunk.
    if current_sentences:
        chunks.append(" ".join(current_sentences).strip())

    return [c for c in chunks if c.strip()]

In [37]:
section_2_2 = next((s for s in sections if "2.2" in s["section_path"]), None)
print(section_2_2['section_path'])

2. Maintenance and Management of Existing Bridges > 2.2 General Administration and Management Systems


In [38]:
print(f"Original block tokens: {count_tokens(section_2_2['content'])}\n")

Original block tokens: 445



In [39]:
# Test split with different token budgets
for max_tokens in [100, 150, 200]:
    chunks = split_block_by_tokens(section_2_2['content'], max_tokens=max_tokens, overlap_tokens=20)
    print(f"Max tokens: {max_tokens} → {len(chunks)} chunk(s)")
    for i, chunk in enumerate(chunks, 1):
        print(f"  Chunk {i}: {count_tokens(chunk)} tokens")
    print()

Max tokens: 100 → 6 chunk(s)
  Chunk 1: 90 tokens
  Chunk 2: 96 tokens
  Chunk 3: 81 tokens
  Chunk 4: 60 tokens
  Chunk 5: 82 tokens
  Chunk 6: 48 tokens

Max tokens: 150 → 4 chunk(s)
  Chunk 1: 132 tokens
  Chunk 2: 135 tokens
  Chunk 3: 141 tokens
  Chunk 4: 48 tokens

Max tokens: 200 → 3 chunk(s)
  Chunk 1: 186 tokens
  Chunk 2: 177 tokens
  Chunk 3: 81 tokens



In [40]:
def build_embedding_text(
    doc_title: str,
    section_path: str,
    chunk_type: str,
    chunk_text: str,
) -> str:
    """
    Prepend structured metadata to the raw chunk text.

    Embedding models are just language models. They encode meaning by
    telling the model "this is from document X, section Y, and it's a table",
    we push the embedding into a part of the vector space that reflects both
    the content and its provenance.  This makes retrieval more precise when
    a query asks about a specific section or document.
    """
    return (
        f"Document: {doc_title}\n"
        f"Section: {section_path or 'Unknown'}\n"
        f"Type: {chunk_type}\n\n"
        f"{chunk_text}"
    )

In [ ]:
def make_chunk_id(source_file: str, section_path: str, chunk_text: str) -> str:
    """
    Produce a stable, unique identifier for a chunk.

    We hash three fields that together uniquely identify a piece of content:
      - source_file   : different files with the same text get different ids
      - section_path  : same text appearing in two sections gets different ids
      - chunk_text    : the actual content

    SHA-256 gives a 64-character hex string.

    Because the hash is deterministic, re-running the chunker on an unchanged
    document produces the exact same ids, making upserts into a vector store
    safe (overwrite, not duplicate).
    """
    raw = f"{source_file}||{section_path}||{chunk_text}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]

In [ ]:
def chunk_markdown_document(
    markdown_text: str,
    source_file: str,
    max_tokens: int = 4000,
    overlap_tokens: int = 40,
) -> List[Chunk]:
    """
    Main chunking pipeline. Combine everything above.
    markdown text to list of `Chunk` objects.
    
    Args:
        markdown_text : The full content of the markdown file as a string.
        source_file   : Path to the source file (used for metadata and the id).
        max_tokens    : Target token budget per chunk.  
        overlap_tokens: How many tokens of context to carry over between
                        consecutive sub-chunks of the same block.

    Returns:
        A list of `Chunk` dataclass instances, ready to embed and store.
    """
    
    if not markdown_text.strip():
        print("Empty document!")
        return []
    
    # Clean doc title
    doc_title = Path(source_file).stem.replace("_", " ").strip()
    
    sections = parse_markdown_sections(markdown_text)
    if not sections:
        print("No section found")
        return []
    
    all_chunks: List[Chunk] = []
    
    for section in sections:
        section_path = section["section_path"] or doc_title
        prev_heading = section["prev_heading"]
        
        # 1. Split the section content into semantic blocks.
        blocks = split_into_blocks(section["content"])

        for block in blocks:
            block = normalize_whitespace(block)
            chunk_type = detect_chunk_type(block)

            # 2. Further split any block that's too large.
            sub_chunks = split_block_by_tokens(
                block,
                max_tokens=max_tokens,
                overlap_tokens=overlap_tokens,
            )

            for sub_chunk in sub_chunks:
                # 3. Build the enriched embedding text.
                embedding_text = build_embedding_text(
                    doc_title=doc_title,
                    section_path=section_path,
                    chunk_type=chunk_type,
                    chunk_text=sub_chunk,
                )

                # 4. Generate a stable chunk id.
                chunk_id = make_chunk_id(source_file, section_path, sub_chunk)

                # 5. Cache the token count so callers don't re-tokenize.
                token_count = count_tokens(embedding_text)

                all_chunks.append(
                    Chunk(
                        chunk_id=chunk_id,
                        doc_title=doc_title,
                        source_file=source_file,
                        section_path=section_path,
                        prev_heading=prev_heading,
                        chunk_type=chunk_type,
                        chunk_text=sub_chunk,
                        embedding_text=embedding_text,
                        token_count=token_count,
                    )
                )

    return all_chunks

In [43]:
file_path = r"D:\Deakin-Data-Science\T2Y3\SIT378 - Team project B\Project\Project 6 - Bridge and Road Crack Detection System\RAG pipeline\processed_documents\Austroads Guide to Bridge Technology Part 7.md"
with open(file_path, "r", encoding="utf-8") as f:
    markdown_text = f.read()
    
chunks = chunk_markdown_document(
    markdown_text=markdown_text,
    source_file=file_path,
)

In [44]:
print(f"\n{'='*60}")
print(f"  Total chunks produced: {len(chunks)}")
print(f"{'='*60}\n")

# Process first 20 chunks
for i, chunk in enumerate(chunks[:20], 1):
    print(f"--- Chunk {i} ---")
    print(f"  id           : {chunk.chunk_id}")
    print(f"  section_path : {chunk.section_path}")
    print(f"  type         : {chunk.chunk_type}")
    print(f"  tokens       : {chunk.token_count}")
    print(f"  text preview : {chunk.chunk_text[:80]!r}")
    print()


  Total chunks produced: 2119

--- Chunk 1 ---
  id           : c1e06a46dd4e982a
  section_path : 1. Introduction > 1.1 Scope
  type         : paragraph
  tokens       : 60
  text preview : 'Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on t'

--- Chunk 2 ---
  id           : 018703b91e2b36b9
  section_path : 1. Introduction > 1.1 Scope
  type         : bullet_list
  tokens       : 51
  text preview : '- bridge inspection\n- load rating and monitoring\n- maintenance, repair and stren'

--- Chunk 3 ---
  id           : 8c22fd42b50a6c53
  section_path : 1. Introduction > 1.1 Scope
  type         : paragraph
  tokens       : 124
  text preview : 'Part 7 provides documentation of the technical issues and practices relevant to '

--- Chunk 4 ---
  id           : 46ef7f81390644b6
  section_path : 1. Introduction > 1.1 Scope
  type         : paragraph
  tokens       : 131
  text preview : 'The focus of Part 7 is on the practical and technical procedures and stra